In [ ]:
# 逐样本可视化
import json
import random
from pathlib import Path
from collections import defaultdict

import matplotlib.pyplot as plt


def load_jsonl(file_path):
    samples = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            samples.append(json.loads(line))
    return samples


# =========================
# 1. 按 id 分组，并配对 is_sdc=0 / is_sdc=1
# =========================
def group_samples_by_id(samples):
    grouped = defaultdict(list)
    for sample in samples:
        grouped[sample.get("id")].append(sample)
    return grouped


def split_sdc_pair(records):
    non_sdc = None
    sdc = None

    for r in records:
        if non_sdc is None and int(r.get("is_sdc", 0)) == 0:
            non_sdc = r
        if sdc is None and int(r.get("is_sdc", 0)) == 1:
            sdc = r

    return non_sdc, sdc


def sample_paired_ids(samples, num_pairs=10, seed=42):
    random.seed(seed)

    grouped = group_samples_by_id(samples)
    paired = []

    for sample_id, recs in grouped.items():
        non_sdc, sdc = split_sdc_pair(recs)
        if non_sdc is not None and sdc is not None:
            paired.append((sample_id, non_sdc, sdc))

    print(f"[Info] total ids: {len(grouped)}")
    print(f"[Info] paired ids with both sdc/non_sdc: {len(paired)}")

    picked = random.sample(paired, min(num_pairs, len(paired)))
    return picked


# =========================
# 2. 提取 records
# =========================
def extract_records(sample):
    mean_std_cos = sample.get("mean_std_cos", {})
    if not isinstance(mean_std_cos, dict):
        return []
    return mean_std_cos.get("records", [])


def group_records_by_step(records):
    step_dict = defaultdict(list)
    for r in records:
        step = r["step"]
        step_dict[step].append(r)

    for step in step_dict:
        step_dict[step] = sorted(
            step_dict[step],
            key=lambda x: (x["src_layer"], x["tgt_layer"])
        )

    return step_dict


# =========================
# 3. 文本辅助
# =========================
def safe_text(s, max_len=120):
    if s is None:
        return ""
    s = str(s).replace("\n", " ").strip()
    if len(s) > max_len:
        s = s[:max_len] + "..."
    return s


def get_fault_info(sample):
    fault = sample.get("fault", None)

    if not isinstance(fault, dict):
        return {
            "module": "N/A",
            "forward": "N/A",
            "before": "N/A",
            "after": "N/A",
        }

    module = fault.get("module", "N/A")
    forward = fault.get("forward", "N/A")
    before = fault.get("before", "N/A")
    after = fault.get("after", "N/A")

    return {
        "module": str(module),
        "forward": forward,
        "before": before,
        "after": after,
    }


# =========================
# 4. 同一 id 的 non_sdc / sdc 在一张图上画
# =========================
def plot_paired_sample_all_steps(sample_id, non_sdc_sample, sdc_sample, out_dir):
    non_sdc_records = extract_records(non_sdc_sample)
    sdc_records = extract_records(sdc_sample)

    if not non_sdc_records or not sdc_records:
        return

    non_sdc_step_dict = group_records_by_step(non_sdc_records)
    sdc_step_dict = group_records_by_step(sdc_records)

    common_steps = sorted(set(non_sdc_step_dict.keys()) & set(sdc_step_dict.keys()))
    if not common_steps:
        return

    sdc_before_score = sdc_sample.get("before_score", None)
    sdc_after_score = sdc_sample.get("after_score", None)
    sdc_dtel_score = sdc_sample.get("dtel_score", None)

    fault_info = get_fault_info(sdc_sample)
    fault_module = fault_info["module"]
    fault_before = fault_info["before"]
    fault_after = fault_info["after"]
    forward = fault_info["forward"]

    question = safe_text(sdc_sample.get("question", ""), max_len=140)
    gt_answer = safe_text(sdc_sample.get("gt_answer", ""), max_len=100)
    clean_answer = safe_text(sdc_sample.get("clean_answer", ""), max_len=100)
    pred_answer = safe_text(sdc_sample.get("pred_answer", ""), max_len=100)

    save_dir = Path(out_dir) / f"id_{sample_id}"
    save_dir.mkdir(parents=True, exist_ok=True)

    for step in common_steps:
        non_sdc_recs = non_sdc_step_dict[step]
        sdc_recs = sdc_step_dict[step]

        n = min(len(non_sdc_recs), len(sdc_recs))
        if n == 0:
            continue

        non_sdc_recs = non_sdc_recs[:n]
        sdc_recs = sdc_recs[:n]

        x_labels = [
            f"({r['src_layer']},{r['tgt_layer']})"
            for r in non_sdc_recs
        ]

        non_sdc_mean_diff = [abs(r["mean_diff"]) for r in non_sdc_recs]
        sdc_mean_diff = [abs(r["mean_diff"]) for r in sdc_recs]

        non_sdc_std_diff = [abs(r["std_diff"]) for r in non_sdc_recs]
        sdc_std_diff = [abs(r["std_diff"]) for r in sdc_recs]

        non_sdc_cos_sim = [r["cos_sim"] for r in non_sdc_recs]
        sdc_cos_sim = [r["cos_sim"] for r in sdc_recs]

        fig, ax1 = plt.subplots(figsize=(18, 8))
        ax2 = ax1.twinx()

        l1 = ax1.plot(
            x_labels, non_sdc_mean_diff,
            marker="o", linewidth=1.8, markersize=4,
            color="tab:blue", linestyle="-",
            label="non_sdc mean_diff"
        )
        l2 = ax1.plot(
            x_labels, sdc_mean_diff,
            marker="o", linewidth=1.8, markersize=4,
            color="tab:blue", linestyle="--",
            label="sdc mean_diff"
        )

        l3 = ax1.plot(
            x_labels, non_sdc_std_diff,
            marker="s", linewidth=1.8, markersize=4,
            color="tab:orange", linestyle="-",
            label="non_sdc std_diff"
        )
        l4 = ax1.plot(
            x_labels, sdc_std_diff,
            marker="s", linewidth=1.8, markersize=4,
            color="tab:orange", linestyle="--",
            label="sdc std_diff"
        )

        l5 = ax2.plot(
            x_labels, non_sdc_cos_sim,
            marker="^", linewidth=1.8, markersize=4,
            color="tab:green", linestyle="-",
            label="non_sdc cos_sim"
        )
        l6 = ax2.plot(
            x_labels, sdc_cos_sim,
            marker="^", linewidth=1.8, markersize=4,
            color="tab:green", linestyle="--",
            label="sdc cos_sim"
        )

        ax1.set_ylim(-0.3, 0.3)
        ax2.set_ylim(0.5, 1.0)

        title = (
            f"id={sample_id} | paired non_sdc vs sdc | step={step} | "
            f"before={sdc_before_score} | after={sdc_after_score} | dtel={sdc_dtel_score:.4f}"
            if sdc_dtel_score is not None else
            f"id={sample_id} | paired non_sdc vs sdc | step={step}"
        )
        ax1.set_title(title, fontsize=12)

        ax1.set_xlabel("(src_layer, tgt_layer)")
        ax1.set_ylabel("mean/std diff")
        ax2.set_ylabel("cos_sim")

        ax1.tick_params(axis="x", rotation=90)
        ax1.grid(True, linestyle="--", alpha=0.35)

        lines = l1 + l2 + l3 + l4 + l5 + l6
        labels = [line.get_label() for line in lines]
        ax1.legend(lines, labels, loc="upper left", fontsize=9)

        ax1.text(
            0.99, 0.97,
            f"fault.module: {fault_module}\n"
            f"fault.forward: {forward}\n"
            f"fault.before: {fault_before}\n"
            f"fault.after: {fault_after}",
            transform=ax1.transAxes,
            fontsize=10,
            ha="right",
            va="top",
            bbox=dict(
                boxstyle="round,pad=0.3",
                facecolor="white",
                alpha=0.85,
                edgecolor="gray"
            )
        )

        # 只展示 sdc 样本信息
        text_info = (
            f"Q: {question}\n"
            f"GT: {gt_answer}\n"
            f"Clean: {clean_answer}\n"
            f"Pred: {pred_answer}"
        )
        fig.text(
            0.01, 0.01,
            text_info,
            fontsize=9,
            ha="left",
            va="bottom",
            wrap=True
        )

        plt.tight_layout(rect=[0, 0.10, 1, 1])

        save_path = save_dir / f"step_{step:03d}.png"
        plt.savefig(save_path, dpi=150)
        plt.close()


# =========================
# 5. 总入口
# =========================
def visualize_paired_samples(
    jsonl_path,
    out_dir="paired_mean_std_cos",
    num_pairs=10,
    seed=42,
):
    samples = load_jsonl(jsonl_path)

    picked_pairs = sample_paired_ids(
        samples,
        num_pairs=num_pairs,
        seed=seed,
    )

    print(f"[Info] total samples: {len(samples)}")
    print(f"[Info] picked paired ids: {len(picked_pairs)}")

    for sample_id, non_sdc_sample, sdc_sample in picked_pairs:
        plot_paired_sample_all_steps(
            sample_id=sample_id,
            non_sdc_sample=non_sdc_sample,
            sdc_sample=sdc_sample,
            out_dir=out_dir,
        )

    print(f"[Done] figures saved to: {out_dir}")


if __name__ == "__main__":
    visualize_paired_samples(
        jsonl_path="/data1/home/dataset_share/cd_data/detect_LingoQA_Qwen_with_sem.jsonl",
        out_dir="pred_mean_std_cos_paired",
        num_pairs=10,
        seed=42,
    )


In [ ]:
# sdc和no_sdc整体可视化
import math
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


def load_jsonl(file_path):
    samples = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            samples.append(json.loads(line))
    return samples


def has_fault(sample):
    return sample.get("fault", None) is not None


def is_valid_fault_sample(sample, after_threshold=1e30):
    fault = sample.get("fault", None)
    if not isinstance(fault, dict):
        return True

    after = fault.get("after", None)
    if after is None:
        return True

    try:
        after_val = float(after)
    except (TypeError, ValueError):
        return True

    if math.isnan(after_val):
        return False

    if after_val > after_threshold:
        return False

    if after_val < -after_threshold:
        return False

    return True


def filter_valid_samples(samples, after_threshold=1e9):
    filtered = [s for s in samples if is_valid_fault_sample(s, after_threshold=after_threshold)]
    skipped = len(samples) - len(filtered)

    print(f"[Info] total samples: {len(samples)}")
    print(f"[Info] valid samples: {len(filtered)}")
    print(f"[Info] skipped samples (fault.after is nan or abs(fault.after) > {after_threshold}): {skipped}")

    return filtered


def format_threshold(threshold):
    return f"{float(threshold):g}"


def parse_dtel_score(sample):
    dtel_score = sample.get("dtel_score", None)
    try:
        dtel_score = float(dtel_score)
    except (TypeError, ValueError):
        return None
    return abs(dtel_score)


def get_group_names(group_mode="ternary", dtel_threshold=0.3):
    if group_mode == "binary":
        return {
            "non_sdc": "non_sdc",
            "sdc": "sdc",
        }

    if group_mode == "ternary":
        thr_str = format_threshold(dtel_threshold)
        return {
            "non_sdc": "non_sdc",
            "lt": f"sdc_dtel_lt_{thr_str}",
            "ge": f"sdc_dtel_ge_{thr_str}",
        }

    if group_mode == "fault_aware":
        return {
            "no_fault": "no_fault",
            "fault_non_sdc": "fault_non_sdc",
            "fault_sdc": "fault_sdc",
        }

    raise ValueError(f"Unsupported group_mode: {group_mode}")


def get_group_order(group_mode="ternary", dtel_threshold=0.3):
    group_names = get_group_names(group_mode=group_mode, dtel_threshold=dtel_threshold)

    if group_mode == "binary":
        return [group_names["non_sdc"], group_names["sdc"]]

    if group_mode == "ternary":
        return [group_names["non_sdc"], group_names["lt"], group_names["ge"]]

    if group_mode == "fault_aware":
        return [
            group_names["no_fault"],
            group_names["fault_non_sdc"],
            group_names["fault_sdc"],
        ]

    raise ValueError(f"Unsupported group_mode: {group_mode}")


def get_sdc_group(sample, group_mode="ternary", dtel_threshold=0.3):
    is_sdc = int(sample.get("is_sdc", 0))
    fault_exists = has_fault(sample)
    group_names = get_group_names(group_mode=group_mode, dtel_threshold=dtel_threshold)

    if group_mode == "binary":
        return group_names["sdc"] if is_sdc == 1 else group_names["non_sdc"]

    if group_mode == "ternary":
        if is_sdc == 0:
            return group_names["non_sdc"]

        dtel_score = parse_dtel_score(sample)
        if dtel_score is not None and dtel_score < dtel_threshold:
            return group_names["lt"]
        else:
            return group_names["ge"]

    if group_mode == "fault_aware":
        if not fault_exists:
            return group_names["no_fault"]

        if is_sdc == 0:
            return group_names["fault_non_sdc"]

        return group_names["fault_sdc"]

    raise ValueError(f"Unsupported group_mode: {group_mode}")


def flatten_records(samples, group_mode="ternary", dtel_threshold=0.3):
    rows = []

    for sample_idx, sample in enumerate(samples):
        sample_id = sample.get("id", None)
        is_sdc = int(sample.get("is_sdc", 0))
        dtel_score = parse_dtel_score(sample)
        sdc_group = get_sdc_group(sample, group_mode=group_mode, dtel_threshold=dtel_threshold)

        mean_std_cos = sample.get("mean_std_cos", {})
        records = mean_std_cos.get("records", [])

        for r in records:
            src = r["src_layer"]
            tgt = r["tgt_layer"]
            step = r["step"]

            rows.append({
                "sample_idx": sample_idx,
                "sample_id": sample_id,
                "is_sdc": is_sdc,
                "has_fault": has_fault(sample),
                "dtel_score": dtel_score,
                "sdc_group": sdc_group,
                "step": step,
                "src_layer": src,
                "tgt_layer": tgt,
                "layer_pair": f"({src},{tgt})",
                "mean_diff": r["mean_diff"],
                "std_diff": r["std_diff"],
                "cos_sim": r["cos_sim"],
                "abs_mean_diff": abs(r["mean_diff"]),
                "abs_std_diff": abs(r["std_diff"]),
            })

    return pd.DataFrame(rows)


def plot_metric_by_layer_pair(df, metric, out_dir, group_mode="ternary", dtel_threshold=0.3):
    Path(out_dir).mkdir(parents=True, exist_ok=True)

    group_order = get_group_order(group_mode=group_mode, dtel_threshold=dtel_threshold)

    grouped = (
        df.groupby(["sdc_group", "src_layer", "tgt_layer", "layer_pair"], as_index=False)[metric]
        .mean()
    )

    layer_pair_order = (
        grouped[["src_layer", "tgt_layer", "layer_pair"]]
        .drop_duplicates()
        .sort_values(["src_layer", "tgt_layer"])["layer_pair"]
        .tolist()
    )

    grouped["layer_pair"] = pd.Categorical(
        grouped["layer_pair"],
        categories=layer_pair_order,
        ordered=True
    )

    grouped["sdc_group"] = pd.Categorical(
        grouped["sdc_group"],
        categories=group_order,
        ordered=True
    )

    grouped = grouped.sort_values(["src_layer", "tgt_layer", "sdc_group"])

    plt.figure(figsize=(14, 6))
    sns.lineplot(
        data=grouped,
        x="layer_pair",
        y=metric,
        hue="sdc_group",
        style="sdc_group",
        hue_order=group_order,
        style_order=group_order,
        markers=True,
        dashes=False,
    )
    plt.xticks(rotation=90)
    plt.xlabel("(src_layer, tgt_layer)")
    plt.ylabel(f"mean {metric}")

    if group_mode == "binary":
        title_suffix = " (binary)"
    elif group_mode == "ternary":
        title_suffix = f" (ternary, dtel_threshold={dtel_threshold})"
    else:
        title_suffix = " (fault_aware)"

    plt.title(f"Average {metric} by layer pair{title_suffix}")
    plt.grid(True, linestyle="--", alpha=0.3)
    plt.tight_layout()
    plt.savefig(Path(out_dir) / f"{metric}_by_layer_pair.png", dpi=150)
    plt.close()


def plot_metric_by_step(df, metric, out_dir, group_mode="ternary", dtel_threshold=0.3):
    Path(out_dir).mkdir(parents=True, exist_ok=True)

    group_order = get_group_order(group_mode=group_mode, dtel_threshold=dtel_threshold)

    grouped = (
        df.groupby(["sdc_group", "step"], as_index=False)[metric]
        .mean()
    )

    grouped["sdc_group"] = pd.Categorical(
        grouped["sdc_group"],
        categories=group_order,
        ordered=True
    )
    grouped = grouped.sort_values(["step", "sdc_group"])

    plt.figure(figsize=(12, 5))
    sns.lineplot(
        data=grouped,
        x="step",
        y=metric,
        hue="sdc_group",
        style="sdc_group",
        hue_order=group_order,
        style_order=group_order,
        markers=True,
        dashes=False,
    )
    plt.xlabel("step")
    plt.ylabel(f"mean {metric}")

    if group_mode == "binary":
        title_suffix = " (binary)"
    elif group_mode == "ternary":
        title_suffix = f" (ternary, dtel_threshold={dtel_threshold})"
    else:
        title_suffix = " (fault_aware)"

    plt.title(f"Average {metric} by decode step{title_suffix}")
    plt.grid(True, linestyle="--", alpha=0.3)
    plt.tight_layout()
    plt.savefig(Path(out_dir) / f"{metric}_by_step.png", dpi=150)
    plt.close()


def plot_metric_heatmap(df, metric, sdc_group, out_dir):
    Path(out_dir).mkdir(parents=True, exist_ok=True)

    sub = df[df["sdc_group"] == sdc_group].copy()
    if len(sub) == 0:
        print(f"[Warn] no data for heatmap: metric={metric}, group={sdc_group}")
        return

    pivot = (
        sub.groupby(["step", "layer_pair"], as_index=False)[metric]
        .mean()
        .pivot(index="step", columns="layer_pair", values=metric)
    )

    ordered_cols = sorted(
        pivot.columns,
        key=lambda s: tuple(map(int, s.strip("()").split(",")))
    )
    pivot = pivot[ordered_cols]

    plt.figure(figsize=(16, 6))

    if metric == "cos_sim":
        sns.heatmap(
            pivot,
            cmap="coolwarm",
            vmin=0,
            vmax=1,
            center=None
        )
    else:
        sns.heatmap(
            pivot,
            cmap="coolwarm",
            center=0
        )

    plt.title(f"{metric} heatmap | group={sdc_group}")
    plt.xlabel("(src_layer, tgt_layer)")
    plt.ylabel("step")
    plt.tight_layout()
    plt.savefig(Path(out_dir) / f"{metric}_heatmap_{sdc_group}.png", dpi=150)
    plt.close()


def plot_metric_distribution(df, metric, out_dir, group_mode="ternary", dtel_threshold=0.3):
    Path(out_dir).mkdir(parents=True, exist_ok=True)

    group_order = get_group_order(group_mode=group_mode, dtel_threshold=dtel_threshold)

    plt.figure(figsize=(9, 5))
    sns.boxplot(
        data=df,
        x="sdc_group",
        y=metric,
        order=group_order
    )

    if group_mode == "binary":
        title_suffix = " (binary)"
    elif group_mode == "ternary":
        title_suffix = f" (ternary, dtel_threshold={dtel_threshold})"
    else:
        title_suffix = " (fault_aware)"

    plt.title(f"Distribution of {metric} by group{title_suffix}")
    plt.grid(True, linestyle="--", alpha=0.3)
    plt.tight_layout()
    plt.savefig(Path(out_dir) / f"{metric}_boxplot.png", dpi=150)
    plt.close()


def visualize_sdc_overall_difference(
    jsonl_path,
    out_dir="viz_overall",
    after_threshold=1e9,
    group_mode="ternary",
    dtel_threshold=0.3
):
    if group_mode not in {"binary", "ternary", "fault_aware"}:
        raise ValueError(
            f"group_mode must be 'binary', 'ternary' or 'fault_aware', got: {group_mode}"
        )

    samples = load_jsonl(jsonl_path)
    samples = filter_valid_samples(samples, after_threshold=after_threshold)
    df = flatten_records(samples, group_mode=group_mode, dtel_threshold=dtel_threshold)

    if len(df) == 0:
        print("[Warn] no valid records found.")
        return

    Path(out_dir).mkdir(parents=True, exist_ok=True)

    group_names = get_group_names(group_mode=group_mode, dtel_threshold=dtel_threshold)
    group_order = get_group_order(group_mode=group_mode, dtel_threshold=dtel_threshold)

    print(f"[Info] group_mode = {group_mode}")
    print(f"[Info] dtel_threshold = {dtel_threshold}")
    print(f"[Info] group names = {group_names}")

    print("[Info] valid sample counts by sdc_group:")
    sample_groups = [get_sdc_group(s, group_mode=group_mode, dtel_threshold=dtel_threshold) for s in samples]
    print(pd.Series(sample_groups).value_counts())

    print("[Info] flattened row counts by sdc_group:")
    print(df["sdc_group"].value_counts())

    print(f"[Info] flattened total rows: {len(df)}")
    print(f"[Info] unique sample_idx count: {df['sample_idx'].nunique()}")
    print(f"[Info] plot group order: {group_order}")

    metrics = ["cos_sim", "abs_mean_diff", "abs_std_diff"]

    for metric in metrics:
        plot_metric_by_layer_pair(df, metric, out_dir, group_mode=group_mode, dtel_threshold=dtel_threshold)
        plot_metric_by_step(df, metric, out_dir, group_mode=group_mode, dtel_threshold=dtel_threshold)

        for group in group_order:
            plot_metric_heatmap(df, metric, sdc_group=group, out_dir=out_dir)

        plot_metric_distribution(df, metric, out_dir, group_mode=group_mode, dtel_threshold=dtel_threshold)

    print(f"[Done] saved all plots to: {out_dir}")


if __name__ == "__main__":
    jsonl_path = "/data1/home/dataset_share/cd_data/Qwen2.5-VL-7B/final/detect_LingoQA_Qwen_with_sem.jsonl"
    # 1) 二分类
    # visualize_sdc_overall_difference(
    #     jsonl_path=jsonl_path,
    #     out_dir="viz_overall_binary",
    #     after_threshold=100,
    #     group_mode="binary",
    #     dtel_threshold=0.5
    # )

    # 2) 基于 dtel 的三分类
    visualize_sdc_overall_difference(
        jsonl_path=jsonl_path,
        out_dir="viz_overall_ternary-0.3",
        after_threshold=100,
        group_mode="ternary",
        dtel_threshold=0.3
    )

    # 3) 你要的新接口：fault-aware 三分类
    # visualize_sdc_overall_difference(
    #     jsonl_path=jsonl_path,
    #     out_dir="viz_overall_fault_aware",
    #     after_threshold=100,
    #     group_mode="fault_aware",
    #     dtel_threshold=0.2
    # )


In [ ]:
import os
import json
import textwrap
import random
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np


# =========================
# 0. 文本与元信息辅助
# =========================
def safe_str(x):
    if x is None:
        return "None"
    if isinstance(x, (dict, list)):
        try:
            return json.dumps(x, ensure_ascii=False)
        except Exception:
            return str(x)
    return str(x)


def shorten_text(text, width=80, max_lines=3):
    s = safe_str(text).replace("\n", " ").strip()
    if not s:
        return "None"

    lines = textwrap.wrap(s, width=width)
    if len(lines) > max_lines:
        lines = lines[:max_lines]
        if len(lines[-1]) >= 3:
            lines[-1] = lines[-1][:-3] + "..."
        else:
            lines[-1] = lines[-1] + "..."
    return "\n".join(lines)


def extract_record_meta(record):
    before_score = record.get("before_score", None)
    after_score = record.get("after_score", None)

    if before_score is not None and after_score is not None:
        try:
            delta_score = float(after_score) - float(before_score)
        except Exception:
            delta_score = record.get("dtel_score", None)
    else:
        delta_score = record.get("dtel_score", None)

    meta = {
        "id": record.get("id", None),
        "is_sdc": record.get("is_sdc", None),
        "before_score": before_score,
        "after_score": after_score,
        "dtel_score": delta_score,
        "fault": record.get("fault", None),
        "question": record.get("question", None),
        "gt_answer": record.get("gt_answer", None),
        "clean_answer": record.get("clean_answer", None),
        "pred_answer": record.get("pred_answer", None),
    }
    return meta


def format_record_text_only_qa(record, role_name="sample"):
    meta = extract_record_meta(record)

    header = (
        f"[{role_name}] "
        f"id={safe_str(meta['id'])} | "
        f"is_sdc={safe_str(meta['is_sdc'])} | "
        f"before={safe_str(meta['before_score'])} | "
        f"after={safe_str(meta['after_score'])} | "
        f"delta={safe_str(meta['dtel_score'])}"
    )

    question_text = shorten_text(meta["question"], width=90, max_lines=3)
    gt_text = shorten_text(meta["gt_answer"], width=90, max_lines=2)
    clean_text = shorten_text(meta["clean_answer"], width=90, max_lines=2)
    pred_text = shorten_text(meta["pred_answer"], width=90, max_lines=2)

    body = (
        f"question: {question_text}\n"
        f"gt_answer: {gt_text}\n"
        f"clean_answer: {clean_text}\n"
        f"pred_answer: {pred_text}"
    )

    return header + "\n" + body


def format_fault_text(record):
    fault = record.get("fault", None)
    if not isinstance(fault, dict):
        return "fault: None"

    lines = []
    for k, v in fault.items():
        lines.append(f"{k}: {safe_str(v)}")
    return "fault\n" + "\n".join(lines)


# =========================
# 1. 读取 jsonl
# =========================
def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            records.append(json.loads(line))
    return records


def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


# =========================
# 2. 按 id 分组，并严格按 is_sdc 配对
# =========================
def group_by_id(records):
    grouped = defaultdict(list)
    for r in records:
        grouped[r["id"]].append(r)
    return grouped


def is_sdc_record(r):
    return r.get("is_sdc", 0) == 1


def is_non_sdc_record(r):
    return r.get("is_sdc", 0) == 0


def split_sdc_pair_by_id(records):
    non_sdc = None
    sdc = None

    for r in records:
        if non_sdc is None and is_non_sdc_record(r):
            non_sdc = r
        if sdc is None and is_sdc_record(r):
            sdc = r

    return non_sdc, sdc


def sample_paired_records(grouped_records, max_pairs=10, seed=42):
    paired = []

    for sample_id, recs in grouped_records.items():
        non_sdc_record, sdc_record = split_sdc_pair_by_id(recs)
        if non_sdc_record is not None and sdc_record is not None:
            paired.append((sample_id, non_sdc_record, sdc_record))

    rng = random.Random(seed)
    if len(paired) > max_pairs:
        paired = rng.sample(paired, max_pairs)

    return paired


# =========================
# 3. interlayer 读取
# =========================
def get_interlayer_feature(record, branch="attn"):
    key = f"interlayer_{branch}"
    return record.get("features", {}).get(key, [])


def get_interlayer_step_metric(record, branch, step_idx, metric):
    blocks = get_interlayer_feature(record, branch=branch)

    if not isinstance(blocks, list):
        return None, None

    if step_idx < 0 or step_idx >= len(blocks):
        return None, None

    step_block = blocks[step_idx]
    values = step_block.get(metric, None)
    pair_labels = step_block.get("layer_pairs", None)

    if not isinstance(values, list):
        return None, None

    if pair_labels is not None and isinstance(pair_labels, list):
        n = min(len(values), len(pair_labels))
        values = values[:n]
        pair_labels = pair_labels[:n]
    else:
        pair_labels = None

    return values, pair_labels


# =========================
# 4. temporal 读取
# =========================
def get_temporal_feature(record, branch="attn"):
    key = f"temporal_{branch}"
    return record.get("features", {}).get(key, {})


def get_temporal_layer_metric(record, branch, layer_idx, metric):
    blocks = get_temporal_feature(record, branch=branch)

    if not isinstance(blocks, dict):
        return None, None

    if str(layer_idx) in blocks:
        layer_block = blocks[str(layer_idx)]
    elif layer_idx in blocks:
        layer_block = blocks[layer_idx]
    else:
        return None, None

    values = layer_block.get(metric, None)
    pair_labels = layer_block.get("step_pairs", None)

    if not isinstance(values, list):
        return None, None

    if pair_labels is not None and isinstance(pair_labels, list):
        n = min(len(values), len(pair_labels))
        values = values[:n]
        pair_labels = pair_labels[:n]
    else:
        pair_labels = None

    return values, pair_labels


# =========================
# 5. layerwise 读取
# =========================
def get_layerwise_feature(record, branch="attn"):
    key = f"layerwise_{branch}"
    return record.get("features", {}).get(key, [])


def get_layerwise_step_metric(record, branch, step_idx, metric):
    blocks = get_layerwise_feature(record, branch=branch)

    if not isinstance(blocks, list):
        return None, None

    if step_idx < 0 or step_idx >= len(blocks):
        return None, None

    step_block = blocks[step_idx]
    values = step_block.get(metric, None)
    layer_labels = step_block.get("layers", None)

    if not isinstance(values, list):
        return None, None

    if layer_labels is not None and isinstance(layer_labels, list):
        n = min(len(values), len(layer_labels))
        values = values[:n]
        layer_labels = layer_labels[:n]
    else:
        layer_labels = None

    return values, layer_labels


# =========================
# 6. 图例统计
# =========================
def format_mean_std_label(name, values):
    arr = np.asarray(values, dtype=np.float32).reshape(-1)
    if arr.size == 0:
        return f"{name} | mean=None | std=None"

    mean_val = float(np.mean(arr))
    std_val = float(np.std(arr))
    return f"{name} | mean={mean_val:.6f} | std={std_val:.6f}"


# =========================
# 7. 通用对比折线图
# =========================
def plot_raw_list_curve(
    non_sdc_vals,
    sdc_vals,
    labels,
    title,
    xlabel,
    ylabel,
    save_path,
    add_delta=True,
    rotate_xticks=False,
    non_sdc_record=None,
    sdc_record=None,
):
    if non_sdc_vals is None or sdc_vals is None:
        return False

    n = min(len(non_sdc_vals), len(sdc_vals))
    if n == 0:
        return False

    non_sdc_vals = np.asarray(non_sdc_vals[:n], dtype=np.float32)
    sdc_vals = np.asarray(sdc_vals[:n], dtype=np.float32)

    if labels is not None:
        labels = labels[:n]

    x = np.arange(n)

    non_sdc_label = format_mean_std_label("non_sdc", non_sdc_vals)
    sdc_label = format_mean_std_label("sdc", sdc_vals)

    fig, ax = plt.subplots(figsize=(14, 9))

    ax.plot(x, non_sdc_vals, marker="o", linewidth=2, label=non_sdc_label)
    ax.plot(x, sdc_vals, marker="o", linewidth=2, label=sdc_label)

    if add_delta:
        delta_vals = sdc_vals - non_sdc_vals
        delta_label = format_mean_std_label("delta(sdc-non_sdc)", delta_vals)
        ax.plot(x, delta_vals, marker="x", linestyle="--", linewidth=1.8, label=delta_label)
        ax.axhline(0, color="gray", linestyle="--", linewidth=1)

    if labels is not None:
        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=90 if rotate_xticks else 0)

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)

    # ===== 右上角标注故障信息（只取 sdc 样本）=====
    if sdc_record is not None:
        fault_text = format_fault_text(sdc_record)
        ax.text(
            0.99, 0.97,
            fault_text,
            transform=ax.transAxes,
            fontsize=9,
            ha="right",
            va="top",
            family="monospace",
            bbox=dict(
                boxstyle="round,pad=0.3",
                facecolor="white",
                alpha=0.88,
                edgecolor="gray"
            )
        )

    # ===== 底部仅保留问答信息 =====
    if sdc_record is not None:
        meta_text = format_record_text_only_qa(sdc_record, role_name="sdc")
        fig.text(
            0.01, 0.01,
            meta_text,
            ha="left",
            va="bottom",
            fontsize=9,
            family="monospace",
            bbox=dict(
                boxstyle="round",
                facecolor="#f7f7f7",
                alpha=0.95,
                edgecolor="#cccccc",
            ),
        )

    plt.tight_layout(rect=[0, 0.22, 1, 1])
    plt.savefig(save_path, dpi=160)
    plt.close(fig)
    return True


# =========================
# 8. interlayer 作图
# =========================
def plot_interlayer_one_step(
    sample_id,
    non_sdc_record,
    sdc_record,
    branch,
    metric,
    step_idx,
    save_path,
    add_delta=True,
):
    non_sdc_vals, non_sdc_pairs = get_interlayer_step_metric(non_sdc_record, branch, step_idx, metric)
    sdc_vals, sdc_pairs = get_interlayer_step_metric(sdc_record, branch, step_idx, metric)

    if non_sdc_vals is None or sdc_vals is None:
        return False

    pair_labels = non_sdc_pairs if non_sdc_pairs is not None else sdc_pairs
    if pair_labels is not None:
        pair_labels = [f"{a}-{b}" for a, b in pair_labels]

    title = f"id={sample_id} | interlayer_{branch} | step={step_idx} | {metric}"

    return plot_raw_list_curve(
        non_sdc_vals=non_sdc_vals,
        sdc_vals=sdc_vals,
        labels=pair_labels,
        title=title,
        xlabel="adjacent layer pair",
        ylabel=metric,
        save_path=save_path,
        add_delta=add_delta,
        rotate_xticks=True,
        non_sdc_record=non_sdc_record,
        sdc_record=sdc_record,
    )


# =========================
# 9. temporal 作图
# =========================
def plot_temporal_one_layer(
    sample_id,
    non_sdc_record,
    sdc_record,
    branch,
    metric,
    layer_idx,
    save_path,
    add_delta=True,
):
    non_sdc_vals, non_sdc_pairs = get_temporal_layer_metric(non_sdc_record, branch, layer_idx, metric)
    sdc_vals, sdc_pairs = get_temporal_layer_metric(sdc_record, branch, layer_idx, metric)

    if non_sdc_vals is None or sdc_vals is None:
        return False

    pair_labels = non_sdc_pairs if non_sdc_pairs is not None else sdc_pairs
    if pair_labels is not None:
        pair_labels = [f"{a}-{b}" for a, b in pair_labels]

    title = f"id={sample_id} | temporal_{branch} | layer={layer_idx} | {metric}"

    return plot_raw_list_curve(
        non_sdc_vals=non_sdc_vals,
        sdc_vals=sdc_vals,
        labels=pair_labels,
        title=title,
        xlabel="adjacent step pair",
        ylabel=metric,
        save_path=save_path,
        add_delta=add_delta,
        rotate_xticks=True,
        non_sdc_record=non_sdc_record,
        sdc_record=sdc_record,
    )


# =========================
# 10. layerwise 作图
# =========================
def plot_layerwise_one_step(
    sample_id,
    non_sdc_record,
    sdc_record,
    branch,
    metric,
    step_idx,
    save_path,
    add_delta=True,
):
    non_sdc_vals, non_sdc_layers = get_layerwise_step_metric(non_sdc_record, branch, step_idx, metric)
    sdc_vals, sdc_layers = get_layerwise_step_metric(sdc_record, branch, step_idx, metric)

    if non_sdc_vals is None or sdc_vals is None:
        return False

    layer_labels = non_sdc_layers if non_sdc_layers is not None else sdc_layers
    if layer_labels is not None:
        layer_labels = [str(x) for x in layer_labels]

    title = f"id={sample_id} | layerwise_{branch} | step={step_idx} | {metric}"

    return plot_raw_list_curve(
        non_sdc_vals=non_sdc_vals,
        sdc_vals=sdc_vals,
        labels=layer_labels,
        title=title,
        xlabel="layer",
        ylabel=metric,
        save_path=save_path,
        add_delta=add_delta,
        rotate_xticks=True,
        non_sdc_record=non_sdc_record,
        sdc_record=sdc_record,
    )


# =========================
# 11. interlayer 批量作图
# =========================
def visualize_interlayer_raw_lists_for_pair(
    sample_id,
    non_sdc_record,
    sdc_record,
    save_root,
    branches=("attn", "mlp"),
    metrics=("cos_sim", "rmse", "diff_mean", "diff_std"),
    add_delta=True,
):
    total_figs = 0

    for branch in branches:
        branch_dir = os.path.join(save_root, f"interlayer_{branch}")
        ensure_dir(branch_dir)

        non_sdc_blocks = get_interlayer_feature(non_sdc_record, branch=branch)
        sdc_blocks = get_interlayer_feature(sdc_record, branch=branch)

        if not isinstance(non_sdc_blocks, list) or not isinstance(sdc_blocks, list):
            continue

        num_steps = min(len(non_sdc_blocks), len(sdc_blocks))

        for step_idx in range(num_steps):
            step_dir = os.path.join(branch_dir, f"step_{step_idx}")
            ensure_dir(step_dir)

            for metric in metrics:
                save_path = os.path.join(
                    step_dir,
                    f"id_{sample_id}__interlayer_{branch}__step_{step_idx}__{metric}.png"
                )

                ok = plot_interlayer_one_step(
                    sample_id=sample_id,
                    non_sdc_record=non_sdc_record,
                    sdc_record=sdc_record,
                    branch=branch,
                    metric=metric,
                    step_idx=step_idx,
                    save_path=save_path,
                    add_delta=add_delta,
                )
                if ok:
                    total_figs += 1

    return total_figs


# =========================
# 12. temporal 批量作图
# =========================
def visualize_temporal_raw_lists_for_pair(
    sample_id,
    non_sdc_record,
    sdc_record,
    save_root,
    branches=("attn", "mlp"),
    metrics=("cos_sim", "rmse", "diff_mean", "diff_std"),
    add_delta=True,
    layer_list=range(28),
):
    total_figs = 0

    for branch in branches:
        branch_dir = os.path.join(save_root, f"temporal_{branch}")
        ensure_dir(branch_dir)

        for layer_idx in layer_list:
            layer_dir = os.path.join(branch_dir, f"layer_{layer_idx}")
            ensure_dir(layer_dir)

            for metric in metrics:
                save_path = os.path.join(
                    layer_dir,
                    f"id_{sample_id}__temporal_{branch}__layer_{layer_idx}__{metric}.png"
                )

                ok = plot_temporal_one_layer(
                    sample_id=sample_id,
                    non_sdc_record=non_sdc_record,
                    sdc_record=sdc_record,
                    branch=branch,
                    metric=metric,
                    layer_idx=layer_idx,
                    save_path=save_path,
                    add_delta=add_delta,
                )
                if ok:
                    total_figs += 1

    return total_figs


# =========================
# 13. layerwise 批量作图
# =========================
def visualize_layerwise_raw_lists_for_pair(
    sample_id,
    non_sdc_record,
    sdc_record,
    save_root,
    branches=("attn", "mlp"),
    metrics=("mean", "std"),
    add_delta=True,
):
    total_figs = 0

    for branch in branches:
        branch_dir = os.path.join(save_root, f"layerwise_{branch}")
        ensure_dir(branch_dir)

        non_sdc_blocks = get_layerwise_feature(non_sdc_record, branch=branch)
        sdc_blocks = get_layerwise_feature(sdc_record, branch=branch)

        if not isinstance(non_sdc_blocks, list) or not isinstance(sdc_blocks, list):
            continue

        num_steps = min(len(non_sdc_blocks), len(sdc_blocks))

        for step_idx in range(num_steps):
            step_dir = os.path.join(branch_dir, f"step_{step_idx}")
            ensure_dir(step_dir)

            for metric in metrics:
                save_path = os.path.join(
                    step_dir,
                    f"id_{sample_id}__layerwise_{branch}__step_{step_idx}__{metric}.png"
                )

                ok = plot_layerwise_one_step(
                    sample_id=sample_id,
                    non_sdc_record=non_sdc_record,
                    sdc_record=sdc_record,
                    branch=branch,
                    metric=metric,
                    step_idx=step_idx,
                    save_path=save_path,
                    add_delta=add_delta,
                )
                if ok:
                    total_figs += 1

    return total_figs


# =========================
# 14. 总入口
# =========================
def visualize_all_raw_lists(
    jsonl_path,
    save_root,
    interlayer_branches=("attn", "mlp"),
    temporal_branches=("attn", "mlp"),
    layerwise_branches=("attn", "mlp"),
    interlayer_metrics=("cos_sim", "rmse", "diff_mean", "diff_std"),
    temporal_metrics=("cos_sim", "rmse", "diff_mean", "diff_std"),
    layerwise_metrics=("mean", "std"),
    add_delta=True,
    layer_list=range(28),
    max_pairs=10,
    seed=42,
):
    records = load_jsonl(jsonl_path)
    grouped = group_by_id(records)
    ensure_dir(save_root)

    paired_records = sample_paired_records(grouped, max_pairs=max_pairs, seed=seed)

    total_pairs = 0
    total_figs = 0

    for sample_id, non_sdc_record, sdc_record in paired_records:
        total_pairs += 1
        sample_dir = os.path.join(save_root, f"id_{sample_id}")
        ensure_dir(sample_dir)

        n1 = visualize_interlayer_raw_lists_for_pair(
            sample_id=sample_id,
            non_sdc_record=non_sdc_record,
            sdc_record=sdc_record,
            save_root=sample_dir,
            branches=interlayer_branches,
            metrics=interlayer_metrics,
            add_delta=add_delta,
        )

        n2 = visualize_temporal_raw_lists_for_pair(
            sample_id=sample_id,
            non_sdc_record=non_sdc_record,
            sdc_record=sdc_record,
            save_root=sample_dir,
            branches=temporal_branches,
            metrics=temporal_metrics,
            add_delta=add_delta,
            layer_list=layer_list,
        )

        n3 = visualize_layerwise_raw_lists_for_pair(
            sample_id=sample_id,
            non_sdc_record=non_sdc_record,
            sdc_record=sdc_record,
            save_root=sample_dir,
            branches=layerwise_branches,
            metrics=layerwise_metrics,
            add_delta=add_delta,
        )

        total_figs += n1 + n2 + n3

    print(f"Done. sampled paired ids = {total_pairs}, figures = {total_figs}")


if __name__ == "__main__":
    jsonl_path = "/data1/home/dataset_share/cd_data/detect_LingoQA_Qwen_with_interlayer.jsonl"
    save_root = "plots_layerwise_lists"

    visualize_all_raw_lists(
        jsonl_path=jsonl_path,
        save_root=save_root,
        interlayer_branches=("attn", "mlp"),
        temporal_branches=("attn", "mlp"),
        layerwise_branches=("attn", "mlp"),
        interlayer_metrics=("cos_sim", "diff_mean", "diff_std"),
        temporal_metrics=("cos_sim", "diff_mean", "diff_std"),
        layerwise_metrics=("mean", "std"),
        add_delta=True,
        layer_list=range(28),
        max_pairs=10,
        seed=42,
    )


In [ ]:
import os
import json
import math
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# =========================
# 1. 基础工具
# =========================
def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            records.append(json.loads(line))
    return records


def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def get_group_name(record):
    return "sdc" if int(record.get("is_sdc", 0)) == 1 else "non_sdc"


def is_valid_fault_sample(record, after_threshold=100):
    fault = record.get("fault", None)

    if not isinstance(fault, dict):
        return True

    after = fault.get("after", None)
    if after is None:
        return True

    try:
        after_val = float(after)
    except (TypeError, ValueError):
        return True

    if math.isnan(after_val):
        return False

    if after_val > after_threshold:
        return False

    if after_val < -after_threshold:
        return False

    return True


def filter_valid_records(records, after_threshold=100):
    filtered = [r for r in records if is_valid_fault_sample(r, after_threshold=after_threshold)]
    skipped = len(records) - len(filtered)

    print(f"[Info] total records: {len(records)}")
    print(f"[Info] valid records: {len(filtered)}")
    print(f"[Info] skipped records (fault.after is nan or abs(fault.after) > {after_threshold}): {skipped}")

    return filtered


# =========================
# 2. 展平 interlayer
# =========================
def flatten_interlayer(records, branch="attn", metrics=("cos_sim", "rmse", "diff_mean", "diff_std")):
    rows = []

    for sample_idx, record in enumerate(records):
        group = get_group_name(record)
        sample_id = record.get("id", None)
        features = record.get("features", {})
        key = f"interlayer_{branch}"
        blocks = features.get(key, [])

        if not isinstance(blocks, list):
            continue

        for step_idx, block in enumerate(blocks):
            layer_pairs = block.get("layer_pairs", None)
            if not isinstance(layer_pairs, list):
                continue

            for metric in metrics:
                values = block.get(metric, None)
                if not isinstance(values, list):
                    continue

                n = min(len(layer_pairs), len(values))
                for i in range(n):
                    pair = layer_pairs[i]
                    val = values[i]

                    if not isinstance(pair, list) and not isinstance(pair, tuple):
                        continue
                    if len(pair) != 2:
                        continue

                    src, tgt = pair
                    rows.append({
                        "sample_idx": sample_idx,
                        "sample_id": sample_id,
                        "group": group,
                        "branch": branch,
                        "metric": metric,
                        "step": step_idx,
                        "src_layer": src,
                        "tgt_layer": tgt,
                        "pair_label": f"{src}-{tgt}",
                        "value": val,
                    })

    return pd.DataFrame(rows)


# =========================
# 3. 展平 temporal
# =========================
def flatten_temporal(records, branch="attn", metrics=("cos_sim", "rmse", "diff_mean", "diff_std")):
    rows = []

    for sample_idx, record in enumerate(records):
        group = get_group_name(record)
        sample_id = record.get("id", None)
        features = record.get("features", {})
        key = f"temporal_{branch}"
        blocks = features.get(key, {})

        if not isinstance(blocks, dict):
            continue

        for layer_key, layer_block in blocks.items():
            try:
                layer_idx = int(layer_key)
            except Exception:
                layer_idx = layer_key

            step_pairs = layer_block.get("step_pairs", None)
            if not isinstance(step_pairs, list):
                continue

            for metric in metrics:
                values = layer_block.get(metric, None)
                if not isinstance(values, list):
                    continue

                n = min(len(step_pairs), len(values))
                for i in range(n):
                    pair = step_pairs[i]
                    val = values[i]

                    if not isinstance(pair, list) and not isinstance(pair, tuple):
                        continue
                    if len(pair) != 2:
                        continue

                    s1, s2 = pair
                    rows.append({
                        "sample_idx": sample_idx,
                        "sample_id": sample_id,
                        "group": group,
                        "branch": branch,
                        "metric": metric,
                        "layer": layer_idx,
                        "step_pair": f"{s1}-{s2}",
                        "step_pair_left": s1,
                        "step_pair_right": s2,
                        "value": val,
                    })

    return pd.DataFrame(rows)


# =========================
# 4. 展平 layerwise
# =========================
def flatten_layerwise(records, branch="attn", metrics=("mean", "std")):
    rows = []

    for sample_idx, record in enumerate(records):
        group = get_group_name(record)
        sample_id = record.get("id", None)
        features = record.get("features", {})
        key = f"layerwise_{branch}"
        blocks = features.get(key, [])

        if not isinstance(blocks, list):
            continue

        for step_idx, block in enumerate(blocks):
            layers = block.get("layers", None)
            if not isinstance(layers, list):
                continue

            for metric in metrics:
                values = block.get(metric, None)
                if not isinstance(values, list):
                    continue

                n = min(len(layers), len(values))
                for i in range(n):
                    rows.append({
                        "sample_idx": sample_idx,
                        "sample_id": sample_id,
                        "group": group,
                        "branch": branch,
                        "metric": metric,
                        "step": step_idx,
                        "layer": layers[i],
                        "value": values[i],
                    })

    return pd.DataFrame(rows)


# =========================
# 5. 绘图：interlayer
# =========================
def plot_interlayer_overall(df, out_dir):
    if len(df) == 0:
        print("[Warn] interlayer dataframe is empty.")
        return

    ensure_dir(out_dir)

    for branch in sorted(df["branch"].dropna().unique()):
        for metric in sorted(df["metric"].dropna().unique()):
            sub = df[(df["branch"] == branch) & (df["metric"] == metric)].copy()
            if len(sub) == 0:
                continue

            grouped_pair = (
                sub.groupby(["group", "src_layer", "tgt_layer", "pair_label"], as_index=False)["value"]
                .mean()
                .sort_values(["src_layer", "tgt_layer", "group"])
            )

            plt.figure(figsize=(14, 6))
            sns.lineplot(
                data=grouped_pair,
                x="pair_label",
                y="value",
                hue="group",
                style="group",
                markers=True,
                dashes=False,
            )
            plt.xticks(rotation=90)
            plt.xlabel("layer pair")
            plt.ylabel(f"mean {metric}")
            plt.title(f"interlayer_{branch} | {metric} | by layer pair")
            plt.grid(True, linestyle="--", alpha=0.3)
            plt.tight_layout()
            plt.savefig(os.path.join(out_dir, f"interlayer_{branch}__{metric}__by_pair.png"), dpi=150)
            plt.close()

            grouped_step = (
                sub.groupby(["group", "step"], as_index=False)["value"]
                .mean()
                .sort_values(["step", "group"])
            )

            plt.figure(figsize=(10, 5))
            sns.lineplot(
                data=grouped_step,
                x="step",
                y="value",
                hue="group",
                style="group",
                markers=True,
                dashes=False,
            )
            plt.xlabel("step")
            plt.ylabel(f"mean {metric}")
            plt.title(f"interlayer_{branch} | {metric} | by step")
            plt.grid(True, linestyle="--", alpha=0.3)
            plt.tight_layout()
            plt.savefig(os.path.join(out_dir, f"interlayer_{branch}__{metric}__by_step.png"), dpi=150)
            plt.close()

            for group_name in ["non_sdc", "sdc"]:
                sub_g = sub[sub["group"] == group_name]
                if len(sub_g) == 0:
                    continue

                pivot = (
                    sub_g.groupby(["step", "pair_label"], as_index=False)["value"]
                    .mean()
                    .pivot(index="step", columns="pair_label", values="value")
                )

                ordered_cols = sorted(
                    pivot.columns,
                    key=lambda s: tuple(map(int, s.split("-")))
                )
                pivot = pivot[ordered_cols]

                plt.figure(figsize=(16, 6))
                if metric == "cos_sim":
                    sns.heatmap(pivot, cmap="coolwarm", vmin=0, vmax=1)
                else:
                    sns.heatmap(pivot, cmap="coolwarm", center=0)
                plt.title(f"interlayer_{branch} | {metric} | heatmap | {group_name}")
                plt.xlabel("layer pair")
                plt.ylabel("step")
                plt.tight_layout()
                plt.savefig(
                    os.path.join(out_dir, f"interlayer_{branch}__{metric}__heatmap__{group_name}.png"),
                    dpi=150
                )
                plt.close()

            plt.figure(figsize=(8, 5))
            sns.boxplot(data=sub, x="group", y="value", order=["non_sdc", "sdc"])
            plt.title(f"interlayer_{branch} | {metric} | distribution")
            plt.grid(True, linestyle="--", alpha=0.3)
            plt.tight_layout()
            plt.savefig(os.path.join(out_dir, f"interlayer_{branch}__{metric}__boxplot.png"), dpi=150)
            plt.close()


# =========================
# 6. 绘图：temporal
# =========================
def plot_temporal_overall(df, out_dir):
    if len(df) == 0:
        print("[Warn] temporal dataframe is empty.")
        return

    ensure_dir(out_dir)

    for branch in sorted(df["branch"].dropna().unique()):
        for metric in sorted(df["metric"].dropna().unique()):
            sub = df[(df["branch"] == branch) & (df["metric"] == metric)].copy()
            if len(sub) == 0:
                continue

            grouped_layer = (
                sub.groupby(["group", "layer"], as_index=False)["value"]
                .mean()
                .sort_values(["layer", "group"])
            )

            plt.figure(figsize=(12, 5))
            sns.lineplot(
                data=grouped_layer,
                x="layer",
                y="value",
                hue="group",
                style="group",
                markers=True,
                dashes=False,
            )
            plt.xlabel("layer")
            plt.ylabel(f"mean {metric}")
            plt.title(f"temporal_{branch} | {metric} | by layer")
            plt.grid(True, linestyle="--", alpha=0.3)
            plt.tight_layout()
            plt.savefig(os.path.join(out_dir, f"temporal_{branch}__{metric}__by_layer.png"), dpi=150)
            plt.close()

            grouped_pair = (
                sub.groupby(["group", "step_pair_left", "step_pair"], as_index=False)["value"]
                .mean()
                .sort_values(["step_pair_left", "group"])
            )

            plt.figure(figsize=(12, 5))
            sns.lineplot(
                data=grouped_pair,
                x="step_pair",
                y="value",
                hue="group",
                style="group",
                markers=True,
                dashes=False,
            )
            plt.xlabel("step pair")
            plt.ylabel(f"mean {metric}")
            plt.title(f"temporal_{branch} | {metric} | by step pair")
            plt.grid(True, linestyle="--", alpha=0.3)
            plt.tight_layout()
            plt.savefig(os.path.join(out_dir, f"temporal_{branch}__{metric}__by_step_pair.png"), dpi=150)
            plt.close()

            for group_name in ["non_sdc", "sdc"]:
                sub_g = sub[sub["group"] == group_name]
                if len(sub_g) == 0:
                    continue

                pivot = (
                    sub_g.groupby(["layer", "step_pair"], as_index=False)["value"]
                    .mean()
                    .pivot(index="layer", columns="step_pair", values="value")
                )

                ordered_cols = sorted(
                    pivot.columns,
                    key=lambda s: tuple(map(int, s.split("-")))
                )
                pivot = pivot[ordered_cols]

                plt.figure(figsize=(14, 6))
                if metric == "cos_sim":
                    sns.heatmap(pivot, cmap="coolwarm", vmin=0, vmax=1)
                else:
                    sns.heatmap(pivot, cmap="coolwarm", center=0)
                plt.title(f"temporal_{branch} | {metric} | heatmap | {group_name}")
                plt.xlabel("step pair")
                plt.ylabel("layer")
                plt.tight_layout()
                plt.savefig(
                    os.path.join(out_dir, f"temporal_{branch}__{metric}__heatmap__{group_name}.png"),
                    dpi=150
                )
                plt.close()

            plt.figure(figsize=(8, 5))
            sns.boxplot(data=sub, x="group", y="value", order=["non_sdc", "sdc"])
            plt.title(f"temporal_{branch} | {metric} | distribution")
            plt.grid(True, linestyle="--", alpha=0.3)
            plt.tight_layout()
            plt.savefig(os.path.join(out_dir, f"temporal_{branch}__{metric}__boxplot.png"), dpi=150)
            plt.close()


# =========================
# 7. 绘图：layerwise
# =========================
def plot_layerwise_overall(df, out_dir):
    if len(df) == 0:
        print("[Warn] layerwise dataframe is empty.")
        return

    ensure_dir(out_dir)

    for branch in sorted(df["branch"].dropna().unique()):
        for metric in sorted(df["metric"].dropna().unique()):
            sub = df[(df["branch"] == branch) & (df["metric"] == metric)].copy()
            if len(sub) == 0:
                continue

            grouped_layer = (
                sub.groupby(["group", "layer"], as_index=False)["value"]
                .mean()
                .sort_values(["layer", "group"])
            )

            plt.figure(figsize=(12, 5))
            sns.lineplot(
                data=grouped_layer,
                x="layer",
                y="value",
                hue="group",
                style="group",
                markers=True,
                dashes=False,
            )
            plt.xlabel("layer")
            plt.ylabel(f"mean {metric}")
            plt.title(f"layerwise_{branch} | {metric} | by layer")
            plt.grid(True, linestyle="--", alpha=0.3)
            plt.tight_layout()
            plt.savefig(os.path.join(out_dir, f"layerwise_{branch}__{metric}__by_layer.png"), dpi=150)
            plt.close()

            grouped_step = (
                sub.groupby(["group", "step"], as_index=False)["value"]
                .mean()
                .sort_values(["step", "group"])
            )

            plt.figure(figsize=(10, 5))
            sns.lineplot(
                data=grouped_step,
                x="step",
                y="value",
                hue="group",
                style="group",
                markers=True,
                dashes=False,
            )
            plt.xlabel("step")
            plt.ylabel(f"mean {metric}")
            plt.title(f"layerwise_{branch} | {metric} | by step")
            plt.grid(True, linestyle="--", alpha=0.3)
            plt.tight_layout()
            plt.savefig(os.path.join(out_dir, f"layerwise_{branch}__{metric}__by_step.png"), dpi=150)
            plt.close()

            for group_name in ["non_sdc", "sdc"]:
                sub_g = sub[sub["group"] == group_name]
                if len(sub_g) == 0:
                    continue

                pivot = (
                    sub_g.groupby(["step", "layer"], as_index=False)["value"]
                    .mean()
                    .pivot(index="step", columns="layer", values="value")
                )

                plt.figure(figsize=(14, 6))
                sns.heatmap(pivot, cmap="coolwarm", center=0)
                plt.title(f"layerwise_{branch} | {metric} | heatmap | {group_name}")
                plt.xlabel("layer")
                plt.ylabel("step")
                plt.tight_layout()
                plt.savefig(
                    os.path.join(out_dir, f"layerwise_{branch}__{metric}__heatmap__{group_name}.png"),
                    dpi=150
                )
                plt.close()

            plt.figure(figsize=(8, 5))
            sns.boxplot(data=sub, x="group", y="value", order=["non_sdc", "sdc"])
            plt.title(f"layerwise_{branch} | {metric} | distribution")
            plt.grid(True, linestyle="--", alpha=0.3)
            plt.tight_layout()
            plt.savefig(os.path.join(out_dir, f"layerwise_{branch}__{metric}__boxplot.png"), dpi=150)
            plt.close()


# =========================
# 8. 总入口
# =========================
def visualize_sdc_vs_non_sdc_overall(
    jsonl_path,
    save_root="viz_sdc_vs_non_sdc_overall",
    after_threshold=100,
    interlayer_branches=("attn", "mlp"),
    temporal_branches=("attn", "mlp"),
    layerwise_branches=("attn", "mlp"),
    interlayer_metrics=("cos_sim", "rmse", "diff_mean", "diff_std"),
    temporal_metrics=("cos_sim", "rmse", "diff_mean", "diff_std"),
    layerwise_metrics=("mean", "std"),
):
    records = load_jsonl(jsonl_path)
    records = filter_valid_records(records, after_threshold=after_threshold)
    ensure_dir(save_root)

    interlayer_df_list = []
    for branch in interlayer_branches:
        df = flatten_interlayer(records, branch=branch, metrics=interlayer_metrics)
        if len(df) > 0:
            interlayer_df_list.append(df)

    if interlayer_df_list:
        interlayer_df = pd.concat(interlayer_df_list, ignore_index=True)
        plot_interlayer_overall(interlayer_df, os.path.join(save_root, "interlayer"))
    else:
        print("[Warn] no interlayer data found.")

    temporal_df_list = []
    for branch in temporal_branches:
        df = flatten_temporal(records, branch=branch, metrics=temporal_metrics)
        if len(df) > 0:
            temporal_df_list.append(df)

    if temporal_df_list:
        temporal_df = pd.concat(temporal_df_list, ignore_index=True)
        plot_temporal_overall(temporal_df, os.path.join(save_root, "temporal"))
    else:
        print("[Warn] no temporal data found.")

    layerwise_df_list = []
    for branch in layerwise_branches:
        df = flatten_layerwise(records, branch=branch, metrics=layerwise_metrics)
        if len(df) > 0:
            layerwise_df_list.append(df)

    if layerwise_df_list:
        layerwise_df = pd.concat(layerwise_df_list, ignore_index=True)
        plot_layerwise_overall(layerwise_df, os.path.join(save_root, "layerwise"))
    else:
        print("[Warn] no layerwise data found.")

    print(f"[Done] saved overall comparison plots to: {save_root}")


if __name__ == "__main__":
    visualize_sdc_vs_non_sdc_overall(
        jsonl_path="/data1/home/dataset_share/cd_data/detect_LingoQA_Qwen_with_interlayer.jsonl",
        save_root="viz_sdc_vs_non_sdc_overall",
        after_threshold=100,
        interlayer_branches=("attn", "mlp"),
        temporal_branches=("attn", "mlp"),
        layerwise_branches=("attn", "mlp"),
        interlayer_metrics=("cos_sim", "diff_mean", "diff_std"),
        temporal_metrics=("cos_sim", "diff_mean", "diff_std"),
        layerwise_metrics=("mean", "std"),
    )


In [ ]:
# 提取xgboost训练的CSV文件
import json
import math
import csv
from collections import defaultdict
import pandas as pd
import numpy as np
from nltk.translate.meteor_score import meteor_score
import nltk
from tqdm import tqdm
from sentence_transformers import CrossEncoder
# =========================
# 配置区域
# =========================
INPUT_JSON = "/data1/home/dataset_share/cd_data/Qwen2.5-VL-7B/final/detect_LingoQA_Qwen_with_sem.jsonl"    
OUTPUT_CSV = "sdc_features.csv"    # 输出特征文件
STEP_GROUP_SIZE = 50  # 越大效果越好，因为能获得所有信息；越小效果越差，因为只能看到一个窗口的特征

COS_PAIRS       = [(6, 7), (23, 24), (24, 25), (25, 26), (26, 27)]
MEAN_DIFF_PAIRS = [(6, 7), (23, 24), (24, 25), (25, 26), (26, 27)]
STD_DIFF_PAIRS  = [(6, 7), (23, 24), (24, 25), (25, 26), (26, 27)]


class SimilarityEvaluator:
    def __init__(self, model_name):
        self.model = CrossEncoder(model_name)

    def score(self, text1, text2):
        return float(self.model.predict([(str(text1), str(text2))])[0])
    
def compute_bleu_and_meteor(reference_sentence, candidate_sentence):
    # 使用 word_tokenize
    reference_tokens = nltk.word_tokenize(reference_sentence.lower())
    candidate_tokens = nltk.word_tokenize(candidate_sentence.lower())

    meteor = meteor_score([reference_tokens], candidate_tokens)
    return meteor

# =========================
# 读取 JSON / JSONL
# =========================
def load_data(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read().strip()

    if not text:
        return []

    # 优先尝试整个文件作为 JSON 解析
    try:
        data = json.loads(text)
        if isinstance(data, list):
            return data
        elif isinstance(data, dict):
            return [data]
    except json.JSONDecodeError:
        pass

    # 否则按 JSONL 解析
    data = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                data.append(json.loads(line))
    return data


# =========================
# 按 step 聚合 records
# =========================
def build_step_pair_map(records):
    """
    构建:
    step_pair_map[step][(src_layer, tgt_layer)] = record
    """
    step_pair_map = defaultdict(dict)
    for r in records:
        step = r["step"]
        pair = (r["src_layer"], r["tgt_layer"])
        step_pair_map[step][pair] = r
    return step_pair_map


def safe_mean(values):
    return float(np.mean(values)) if values else np.nan

def safe_min(values):
    return float(np.min(values)) if values else np.nan

def safe_max(values):
    return float(np.max(values)) if values else np.nan

def extract_group_features(step_pair_map, step_list):
    """
    对一个 step 分组提取 20 个特征：
    - cos_sim: 6个pair * (mean, min) = 12
    - mean_diff: 2个pair * (mean, max) = 4
    - std_diff: 2个pair * (mean, max) = 4
    """
    feat = {}

    # 1) cos_sim
    for pair in COS_PAIRS:
        values = []
        for step in step_list:
            rec = step_pair_map.get(step, {}).get(pair)
            if rec is not None and "cos_sim" in rec:
                values.append(rec["cos_sim"])

        feat[f"cos_sim_mean_{pair[0]}_{pair[1]}"] = safe_mean(values)
        feat[f"cos_sim_min_{pair[0]}_{pair[1]}"] = safe_min(values)
        feat[f"cos_sim_range_{pair[0]}_{pair[1]}"] = (
            safe_max(values) - safe_min(values) if len(values) > 0 else np.nan
        )

    # 2) mean_diff
    for pair in MEAN_DIFF_PAIRS:
        raw_values = []
        abs_values = []
        for step in step_list:
            rec = step_pair_map.get(step, {}).get(pair)
            if rec is not None and "mean_diff" in rec:
                v = rec["mean_diff"]
                raw_values.append(v)
                abs_values.append(abs(v))

        feat[f"mean_diff_mean_{pair[0]}_{pair[1]}"] = safe_mean(abs_values)
        feat[f"mean_diff_max_{pair[0]}_{pair[1]}"] = safe_max(abs_values)
        feat[f"mean_diff_range_{pair[0]}_{pair[1]}"] = (
            safe_max(raw_values) - safe_min(raw_values) if len(raw_values) > 0 else np.nan
        )

    # 3) std_diff
    for pair in STD_DIFF_PAIRS:
        raw_values = []
        abs_values = []
        for step in step_list:
            rec = step_pair_map.get(step, {}).get(pair)
            if rec is not None and "std_diff" in rec:
                v = rec["std_diff"]
                raw_values.append(v)
                abs_values.append(abs(v))

        feat[f"std_diff_mean_{pair[0]}_{pair[1]}"] = safe_mean(abs_values)
        feat[f"std_diff_max_{pair[0]}_{pair[1]}"] = safe_max(abs_values)
        feat[f"std_diff_range_{pair[0]}_{pair[1]}"] = (
            safe_max(raw_values) - safe_min(raw_values) if len(raw_values) > 0 else np.nan
        )

    return feat


def extract_features_from_sample(sample, sample_idx, step_group_size=6, threshold=0.3):
    """
    一个原始样本 -> 多个分组样本
    """
    records = sample.get("mean_std_cos", {}).get("records", [])
    if not records:
        return []

    dtel_score = abs(sample.get("dtel_score", None))
    gt_answer = str(sample.get("gt_answer", None))
    clean_answer = str(sample.get("clean_answer", None))
    pred_answer = str(sample.get("pred_answer", None))

    # 打标签的艺术！！！！！！
    dtel_semantics = sample.get("dtel_semantics", None)
    # dtel_semantics = abs(se.score(clean_answer, clean_answer) - se.score(clean_answer, pred_answer))
    label = 0
    if dtel_semantics == 0: 
        label = 0
    elif dtel_semantics <= 0.5:
        label = 1
    else:
        label = 2
    #先按照cross-encoder打标签
    # label = 0
    # if dtel_score > 0 and dtel_score < threshold:
    #     label = 1
    # elif dtel_score >= threshold:
    #     label = 2
    
    # #METEOR作为辅助
    # meteor = compute_bleu_and_meteor(clean_answer, pred_answer)
    # if sim_score > 0.5 and meteor > 0.4:
    #     label = 1

    # #给较大偏差数据打标签
    if dtel_score != 0:
        fault = sample.get("fault", None)
        if fault is not None:
            after = float(fault.get("after", None))
            if math.isnan(after) or pred_answer.startswith("!!!!!!") or pred_answer.endswith("!!!!!!"):
                label = 2
    
    # label = sample.get("is_sdc", None)
    orig_id = sample.get("id", None)

    # 保证每条原始 sample 都有唯一标识
    sample_uid = f"{orig_id}_{sample_idx}"

    step_pair_map = build_step_pair_map(records)
    all_steps = sorted(step_pair_map.keys())

    rows = []
    for group_idx, start in enumerate(range(0, len(all_steps), step_group_size)):
        step_group = all_steps[start:start + step_group_size]
        feat = extract_group_features(step_pair_map, step_group)

        group_uid = f"{sample_uid}_g{group_idx}"

        row = {
            "orig_id": orig_id,
            "sample_uid": sample_uid,
            "group_id": group_idx,         # 局部编号
            "group_uid": group_uid,        # 全局唯一编号
            "step_start": step_group[0],
            "step_end": step_group[-1],
            "num_steps_in_group": len(step_group),
            **feat,
            "label": label
        }
        rows.append(row)

    return rows


# =========================
# 主流程
# =========================
def main():
    data = load_data(INPUT_JSON)
    all_rows = []

    for sample_idx, sample in enumerate(data):
        rows = extract_features_from_sample(
            sample,
            sample_idx=sample_idx,
            step_group_size=STEP_GROUP_SIZE,
            threshold=0.5
        )
        if rows is not None: 
            all_rows.extend(rows)

    df = pd.DataFrame(all_rows)

    feature_cols = [
        c for c in df.columns
        if c not in ["orig_id", "sample_uid", "group_id", "group_uid",
                     "step_start", "step_end", "num_steps_in_group", "label"]
    ]

    print("特征数 =", len(feature_cols))
    print("特征列 =", feature_cols)

    print("sample_uid 是否唯一按原始 sample 区分：", df["orig_id"].nunique())
    print("group_uid 是否全局唯一：", df["group_uid"].is_unique)

    # 按 orig_id 切分
    orig_id_num = pd.to_numeric(df["orig_id"], errors="coerce")
    if orig_id_num.isna().any():
        bad_count = int(orig_id_num.isna().sum())
        raise ValueError(f"orig_id 中有 {bad_count} 个值无法转成数值，不能按阈值切分")

    train_df = df[orig_id_num < 4250].copy()
    valid_df = df[orig_id_num >= 4250].copy()

    train_orig_ids = set(train_df["orig_id"].unique())
    valid_orig_ids = set(valid_df["orig_id"].unique())
    overlap = train_orig_ids.intersection(valid_orig_ids)

    assert len(overlap) == 0, "train 和 valid 的 orig_id 有重叠，存在泄漏风险"

    train_df.to_csv("./train_data/Qwen2.5_LingoQA_train_set.csv", index=False, encoding="utf-8-sig")
    valid_df.to_csv("./train_data/Qwen2.5_LingoQA_valid_set.csv", index=False, encoding="utf-8-sig")

    print("=" * 60)
    print(f"train_set.csv 行数: {len(train_df)}")
    print(f"valid_set.csv 行数: {len(valid_df)}")
    print(f"train orig_id unique: {train_df['orig_id'].nunique()}")
    print(f"valid orig_id unique: {valid_df['orig_id'].nunique()}")
    print(f"train/valid orig_id overlap: {len(overlap)}")
    print("=" * 60)

if __name__ == "__main__":
    x=0
    se = SimilarityEvaluator("/data0/home/lc/cd/stsb-roberta-base")
    main()


In [ ]:
# from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
# from nltk.translate.meteor_score import meteor_score
# import nltk

# # 第一次运行时取消注释下载
# # nltk.download('wordnet')
# # nltk.download('omw-1.4')


# def compute_bleu_and_meteor(reference_sentence, candidate_sentence):
#     # 使用 word_tokenize 而不是 .split()
#     reference_tokens = nltk.word_tokenize(reference_sentence.lower())
#     candidate_tokens = nltk.word_tokenize(candidate_sentence.lower())

#     # METEOR (注意：NLTK 的 meteor_score 要求参考集是 tokens 的列表)
#     meteor = meteor_score([reference_tokens], candidate_tokens)
    
#     # BLEU
#     smoothie = SmoothingFunction().method1
#     bleu = sentence_bleu([reference_tokens], candidate_tokens, smoothing_function=smoothie)

#     return bleu, meteor

# if __name__ == "__main__":
#     reference = "Pedestrians crossing, traffic light green, safe to proceed."
#     candidate = "Pedestrians crossing, traffic light green, safe passage allowed."

#     bleu, meteor = compute_bleu_and_meteor(reference, candidate)

#     print(f"Reference: {reference}")
#     print(f"Candidate: {candidate}")
#     print(f"BLEU:   {bleu:.6f}")
#     print(f"METEOR: {meteor:.6f}")

import json


def filter_jsonl_by_src_layer(
    input_path: str,
    output_path: str,
    keep_src_layers=(6, 24, 25, 26),
):
    keep_src_layers = set(keep_src_layers)

    total = 0
    kept = 0
    skipped_bad_lines = 0

    with open(input_path, "r", encoding="utf-8") as fin, \
         open(output_path, "w", encoding="utf-8") as fout:

        for line in fin:
            line = line.strip()
            if not line:
                continue

            total += 1

            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                skipped_bad_lines += 1
                continue

            if obj.get("src_layer") in keep_src_layers:
                fout.write(json.dumps(obj, ensure_ascii=False) + "\n")
                kept += 1

    print(f"Done.")
    print(f"Total lines: {total}")
    print(f"Kept lines: {kept}")
    print(f"Filtered ratio: {kept / total:.4f}" if total > 0 else "Filtered ratio: N/A")
    print(f"Bad lines skipped: {skipped_bad_lines}")


if __name__ == "__main__":
    filter_jsonl_by_src_layer(
        input_path="attn_proj_interlayer.jsonl",
        output_path="attn_proj_interlayer_src_6_24_25_26.jsonl",
        keep_src_layers=(6, 24, 25, 26),
    )



In [ ]:
import json
from tqdm import tqdm
from sentence_transformers import CrossEncoder


class SimilarityEvaluator:
    def __init__(self, model_name):
        self.model = CrossEncoder(model_name)

    def score(self, text1, text2):
        return float(self.model.predict([(str(text1), str(text2))])[0])


def process_jsonl(
    input_path,
    output_path,
    model_name,
    clean_key="clean_answer",
    pred_key="pred_answer",
    output_key="dtel_semantics",
):
    se = SimilarityEvaluator(model_name)

    with open(input_path, "r", encoding="utf-8") as fin, \
         open(output_path, "w", encoding="utf-8") as fout:

        for line_idx, line in enumerate(tqdm(fin, desc="Processing jsonl", unit="lines"), start=1):
            line = line.strip()
            if not line:
                continue

            obj = json.loads(line)

            if clean_key not in obj:
                raise KeyError(f"Line {line_idx}: missing key '{clean_key}'")

            if pred_key not in obj:
                raise KeyError(f"Line {line_idx}: missing key '{pred_key}'")

            if "gt_answer" not in obj:
                raise KeyError(f"Line {line_idx}: missing key 'gt_answer'")

            gt_answer = obj["gt_answer"]
            clean_answer = obj[clean_key]
            pred_answer = obj[pred_key]

            sim_score = abs(se.score(gt_answer, pred_answer) - se.score(gt_answer, clean_answer))

            # 让 output_key 排在最前面
            new_obj = {output_key: sim_score, **obj}

            fout.write(json.dumps(new_obj, ensure_ascii=False) + "\n")

    print(f"[Info] Done. Output saved to: {output_path}")


def main():
    process_jsonl(
        input_path="/data1/home/dataset_share/cd_data/detect_LingoQA_Qwen_with_cos.jsonl",
        output_path="/data1/home/dataset_share/cd_data/detect_LingoQA_Qwen_with_sem.jsonl",
        model_name="/data0/home/lc/cd/stsb-roberta-base",
        clean_key="clean_answer",
        pred_key="pred_answer",
        output_key="dtel_semantics",
    )


if __name__ == "__main__":
    main()
